In [1]:
import pandas as pd
import numpy as np

# 1. Generate the unique SKUs starting from SKU001 up to SKU050
# By using range(1, 51), we start at 1 and stop before 51
a = [f"SKU{i:03d}" for i in range(1, 51)] 
df = pd.DataFrame(a, columns=['SKU'])

# Set the random seed so your results are identical every time you run it
np.random.seed(42)

# 2. Add Stock Levels (Integers between 0 and 500)
df['Stock_Level'] = np.random.randint(low=0, high=501, size=50)

# 3. Add Categories (Randomly selected text labels from a defined list)
categories_list = ['Electronics', 'Apparel', 'Home & Kitchen', 'Beauty & Health', 'Sports & Outdoors']
df['Category'] = np.random.choice(categories_list, size=50)

# 4. Add Unit Costs (Decimal prices between $5.00 and $200.00, rounded to 2 decimal places)
df['Unit_Cost'] = np.round(np.random.uniform(low=5.00, high=200.00, size=50), 2)

# 5. Add Lead Times (Integer days between 3 and 21 days for supplier shipping)
df['Lead_Time'] = np.random.randint(low=3, high=22, size=50)

# Display the first 5 rows of your completed DataFrame
print(df.head())

    


      SKU  Stock_Level         Category  Unit_Cost  Lead_Time
0  SKU001          102   Home & Kitchen      58.82         14
1  SKU002          435  Beauty & Health      62.77         14
2  SKU003          348  Beauty & Health      37.23         13
3  SKU004          270      Electronics       8.05          9
4  SKU005          106   Home & Kitchen      87.56          3


In [2]:
dates = pd.date_range(start="2024-01-01", end="2025-12-31", freq="D")

sku_grid, date_grid = np.meshgrid(df['SKU'], dates, indexing='ij')

sales_df = pd.DataFrame({'SKU': sku_grid.ravel(), 'Date': date_grid.ravel()})

sales_df['Month'] = sales_df['Date'].dt.month
sales_df['Day_of_Week'] = sales_df['Date'].dt.dayofweek

np.random.seed(42)
base_demand = np.random.randint(low=5, high=16, size=len(sales_df))

multipliers = np.ones(len(sales_df))

multipliers = np.where(sales_df['Day_of_Week'] >= 4, multipliers * 1.5, multipliers)

multipliers = np.where(sales_df['Month'] == 12, multipliers * 2.0, multipliers)


final_sales = (base_demand * multipliers)

np.round(final_sales)

sales_df['Quantity_Sold'] = np.round(final_sales).astype(int)

print(sales_df.head(10))

      SKU       Date  Month  Day_of_Week  Quantity_Sold
0  SKU001 2024-01-01      1            0             11
1  SKU001 2024-01-02      1            1              8
2  SKU001 2024-01-03      1            2             15
3  SKU001 2024-01-04      1            3             12
4  SKU001 2024-01-05      1            4             14
5  SKU001 2024-01-06      1            5             16
6  SKU001 2024-01-07      1            6             21
7  SKU001 2024-01-08      1            0              7
8  SKU001 2024-01-09      1            1             11
9  SKU001 2024-01-10      1            2             15


In [3]:
sales_df.groupby('SKU')['Quantity_Sold'].max()

sales_df.groupby('SKU')['Quantity_Sold'].mean()

safety_stock = (sales_df.groupby('SKU')['Quantity_Sold'].max() - sales_df.groupby('SKU')['Quantity_Sold'].mean()) * 5

reorder_point = sales_df.groupby('SKU')['Quantity_Sold'].mean() * 5 + safety_stock

sales_df['Reorder_Point_Threshold'] = sales_df['SKU'].map(reorder_point)
sales_df['Safety_Stock_Threshold'] = sales_df['SKU'].map(safety_stock)

print(sales_df[['SKU', 'Safety_Stock_Threshold', 'Reorder_Point_Threshold']])

          SKU  Safety_Stock_Threshold  Reorder_Point_Threshold
0      SKU001              145.362517                    210.0
1      SKU001              145.362517                    210.0
2      SKU001              145.362517                    210.0
3      SKU001              145.362517                    210.0
4      SKU001              145.362517                    210.0
...       ...                     ...                      ...
36545  SKU050              158.522572                    225.0
36546  SKU050              158.522572                    225.0
36547  SKU050              158.522572                    225.0
36548  SKU050              158.522572                    225.0
36549  SKU050              158.522572                    225.0

[36550 rows x 3 columns]
